# Roller Coaster – Extended Solution

Complete reference solution with alternate implementations, pattern analysis, and simulation.  
Use the companion **Practice Skeleton** notebook to try the exercises yourself first.

#### Overview

This is an **extended** version of the classic Codecademy Roller Coaster visualization lab.  
You will load Golden Ticket Award rankings (wood & steel, 2013–2018) and a large roller-coaster statistics dataset, build reusable plotting functions, discover patterns (manufacturer specialties, seating-type effects, height–speed relationship), try alternate coding approaches, and run a small simulation by changing key parameters.


#### Project Goals

1. Master pandas filtering + matplotlib/seaborn line, bar, hist, pie and scatter plots.  
2. Write parameterized functions for ranking trends of 1 / 2 / top-n coasters.  
3. Explore real coaster statistics (speed, height, length, inversions).  
4. Identify patterns: popular seating types, manufacturer focus areas, park specialties.  
5. Practice alternate implementations and a parameter-driven simulation.  
6. Produce clear visual insights and a 1-page summary report.


## Prerequisites

You should be comfortable with the first lessons of **Data Analysis with Pandas** and **Data Visualization in Python** (matplotlib). Knowledge of `groupby`, filtering with boolean masks, and basic function writing will help.


## Analysis Pipeline Flowchart

![Roller Coaster Analysis Flowchart](roller_coaster_flowchart.png)

The flowchart above outlines the extended analysis pipeline: load rankings & coaster stats → ranking-over-time visualizations → distributions & categorical plots → pattern mining (manufacturers, seating, parks) → alternate implementations → simulation of parameters → insights & report.


## 1. Load Rankings Data

Load both Golden Ticket CSVs and inspect them.

In [ ]:
# 1
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# load rankings data
wood_rankings = pd.read_csv('Golden_Ticket_Award_Winners_Wood.csv')
steel_rankings = pd.read_csv('Golden_Ticket_Award_Winners_Steel.csv')

print('Wood rankings shape:', wood_rankings.shape)
print(wood_rankings.head())
print('\nSteel rankings shape:', steel_rankings.shape)
print(steel_rankings.head())
print('\nColumns:', wood_rankings.columns.tolist())
print('Years present (wood):', sorted(wood_rankings['Year of Rank'].unique()))


## 2. Ranking of One Coaster Over Time

The same name can appear for different parks (or with slight spelling variations). Adding an optional `park_name` argument solves the ambiguity.

In [ ]:
# 2
def plot_coaster_ranking(coaster_name, rankings_df, park_name=None):
    """Plot ranking trajectory of a single coaster over the years."""
    df = rankings_df[rankings_df['Name'] == coaster_name].copy()
    if park_name is not None:
        df = df[df['Park'] == park_name]
    if df.empty:
        print(f'No data found for {coaster_name}' + (f' at {park_name}' if park_name else ''))
        return
    df = df.sort_values('Year of Rank')
    plt.figure(figsize=(8, 5))
    plt.plot(df['Year of Rank'], df['Rank'], marker='o', linewidth=2, label=coaster_name)
    plt.gca().invert_yaxis()  # rank 1 at top
    plt.title(f'Ranking of {coaster_name} Over Time')
    plt.xlabel('Year')
    plt.ylabel('Rank (1 = best)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close()

# El Toro appears for Six Flags Great Adventure
plot_coaster_ranking('El Toro', wood_rankings, park_name='Six Flags Great Adventure')


## 3. Ranking of Two Coasters Over Time

In [ ]:
# 3
def plot_two_coasters(name1, name2, rankings_df, park1=None, park2=None):
    """Plot ranking trajectories of two coasters on the same axes."""
    def _subset(name, park):
        s = rankings_df[rankings_df['Name'] == name]
        if park is not None:
            s = s[s['Park'] == park]
        return s.sort_values('Year of Rank')

    d1 = _subset(name1, park1)
    d2 = _subset(name2, park2)
    plt.figure(figsize=(9, 5))
    if not d1.empty:
        plt.plot(d1['Year of Rank'], d1['Rank'], marker='o', label=name1)
    if not d2.empty:
        plt.plot(d2['Year of Rank'], d2['Rank'], marker='s', label=name2)
    plt.gca().invert_yaxis()
    plt.title(f'{name1} vs {name2} Rankings Over Time')
    plt.xlabel('Year')
    plt.ylabel('Rank (1 = best)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close()

plot_two_coasters('El Toro', 'Boulder Dash', wood_rankings,
                  park1='Six Flags Great Adventure', park2='Lake Compounce')


## 4. Top-n Rankings Over Time

Any coaster that ever reached rank ≤ n is included; its full history is drawn.

In [ ]:
# 4
def plot_top_n(rankings_df, n=5, title_suffix=''):
    """Plot every coaster that achieved rank <= n at least once."""
    top = rankings_df[rankings_df['Rank'] <= n]
    names = top['Name'].unique()
    plt.figure(figsize=(11, 6))
    for name in names:
        subset = rankings_df[rankings_df['Name'] == name].sort_values('Year of Rank')
        if subset['Park'].nunique() > 1:
            good_park = top[top['Name'] == name]['Park'].iloc[0]
            subset = subset[subset['Park'] == good_park]
        plt.plot(subset['Year of Rank'], subset['Rank'], marker='o', label=name)
    plt.gca().invert_yaxis()
    plt.title(f'Top-{n} Ranked Coasters Over Time{title_suffix}')
    plt.xlabel('Year')
    plt.ylabel('Rank (1 = best)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close()

plot_top_n(wood_rankings, n=5, title_suffix=' (Wood)')
plot_top_n(steel_rankings, n=3, title_suffix=' (Steel)')


## 5. Load Roller-Coaster Statistics

In [ ]:
# 5
roller_coasters = pd.read_csv('roller_coasters.csv')
print('Shape:', roller_coasters.shape)
print('Columns:', roller_coasters.columns.tolist())
print('\nMissing values:')
print(roller_coasters.isna().sum())
print('\nStatus distribution:')
print(roller_coasters['status'].value_counts())
print('\nMaterial types:')
print(roller_coasters['material_type'].value_counts())
print('\nTop seating types:')
print(roller_coasters['seating_type'].value_counts().head(8))
print('\nNumeric summary:')
print(roller_coasters[['speed', 'height', 'length', 'num_inversions']].describe().round(1))


## 6. Histograms

In [ ]:
# 6
def plot_histogram(df, column, bins=30, color='#3498db'):
    data = df[column].dropna()
    plt.figure(figsize=(8, 4.5))
    plt.hist(data, bins=bins, color=color, edgecolor='white', alpha=0.85)
    plt.title(f'Distribution of Roller Coaster {column.replace("_", " ").title()}')
    plt.xlabel(column)
    plt.ylabel('Count')
    plt.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close()

plot_histogram(roller_coasters, 'speed')
plot_histogram(roller_coasters, 'length', color='#9b59b6')
plot_histogram(roller_coasters, 'num_inversions', bins=15, color='#e67e22')

def plot_height_histogram(df, max_height=140):
    data = df.loc[df['height'] < max_height, 'height'].dropna()
    plt.figure(figsize=(8, 4.5))
    plt.hist(data, bins=30, color='#1abc9c', edgecolor='white', alpha=0.85)
    plt.title(f'Distribution of Roller Coaster Height (filtered < {max_height} m)')
    plt.xlabel('height (m)')
    plt.ylabel('Count')
    plt.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close()

plot_height_histogram(roller_coasters)


## 7. Inversions by Coaster at a Park

In [ ]:
# 7
def plot_inversions_by_coaster(df, park_name):
    park_df = df[df['park'] == park_name].dropna(subset=['num_inversions'])
    if park_df.empty:
        print(f'No coasters found for park: {park_name}')
        return
    park_df = park_df.sort_values('num_inversions', ascending=False)
    plt.figure(figsize=(10, 5))
    plt.bar(park_df['name'], park_df['num_inversions'], color='#e74c3c', edgecolor='white')
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.title(f'Number of Inversions by Coaster – {park_name}')
    plt.xlabel('Coaster')
    plt.ylabel('Number of Inversions')
    plt.tight_layout()
    plt.show()
    plt.close()

plot_inversions_by_coaster(roller_coasters, 'Parc Asterix')
plot_inversions_by_coaster(roller_coasters, 'Cedar Point')


## 8. Operating vs Closed Pie

In [ ]:
# 8
def plot_status_pie(df):
    operating = (df['status'] == 'status.operating').sum()
    closed = (df['status'] == 'status.closed.definitely').sum()
    other = len(df) - operating - closed
    sizes = [operating, closed, other]
    labels = ['Operating', 'Closed Definitely', 'Other / Unknown']
    colors = ['#27ae60', '#c0392b', '#95a5a6']
    plt.figure(figsize=(7, 7))
    plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, explode=(0.02, 0.02, 0))
    plt.title('Roller Coaster Operating Status')
    plt.tight_layout()
    plt.show()
    plt.close()

plot_status_pie(roller_coasters)


## 9. Scatter Plots

In [ ]:
# 9
def plot_scatter(df, column_x, column_y):
    clean = df[[column_x, column_y]].dropna()
    plt.figure(figsize=(8, 5))
    plt.scatter(clean[column_x], clean[column_y], alpha=0.45, s=25, c='#2980b9')
    plt.title(f'{column_y} vs {column_x}')
    plt.xlabel(column_x)
    plt.ylabel(column_y)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close()

def plot_speed_vs_height(df, max_height=140):
    clean = df[(df['height'] < max_height) & df['height'].notna() & df['speed'].notna()]
    plt.figure(figsize=(8, 5))
    plt.scatter(clean['height'], clean['speed'], alpha=0.4, s=30, c='#8e44ad')
    if len(clean) > 2:
        z = np.polyfit(clean['height'], clean['speed'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(clean['height'].min(), clean['height'].max(), 100)
        plt.plot(x_line, p(x_line), 'r--', lw=2, label=f'trend (slope={z[0]:.2f})')
        plt.legend()
    plt.title('Speed vs Height (height < 140 m)')
    plt.xlabel('Height (m)')
    plt.ylabel('Speed')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close()
    corr = clean[['height', 'speed']].corr().iloc[0, 1]
    print(f'Pearson correlation (height, speed): {corr:.3f}')

plot_scatter(roller_coasters, 'length', 'speed')
plot_speed_vs_height(roller_coasters)


## More Practice – Pattern Discovery & Insights

In [ ]:
# A – Seating type popularity and performance
print('=== Seating Type Counts ===')
print(roller_coasters['seating_type'].value_counts().head(10))

seat_stats = (roller_coasters
              .groupby('seating_type')
              .agg(count=('name', 'count'),
                   avg_speed=('speed', 'mean'),
                   avg_height=('height', 'mean'),
                   avg_length=('length', 'mean'),
                   avg_inv=('num_inversions', 'mean'))
              .query('count >= 10')
              .sort_values('avg_speed', ascending=False)
              .round(1))
print('\n=== Seating types with ≥10 coasters (by avg speed) ===')
print(seat_stats)

top_seats = roller_coasters['seating_type'].value_counts().head(8)
plt.figure(figsize=(9, 4.5))
plt.barh(top_seats.index[::-1], top_seats.values[::-1], color='#3498db')
plt.title('Most Common Seating Types')
plt.xlabel('Number of Coasters')
plt.tight_layout()
plt.show()
plt.close()


In [ ]:
# B – Manufacturer specialties
mfr_stats = (roller_coasters
             .groupby('manufacturer')
             .agg(count=('name', 'count'),
                  avg_speed=('speed', 'mean'),
                  avg_height=('height', 'mean'),
                  avg_inv=('num_inversions', 'mean'))
             .query('count >= 10')
             .sort_values('avg_speed', ascending=False)
             .round(1))
print('=== Manufacturers with ≥10 coasters (sorted by avg speed) ===')
print(mfr_stats.head(12))

top8 = mfr_stats.head(8)
plt.figure(figsize=(9, 4.5))
plt.barh(top8.index[::-1], top8['avg_speed'][::-1], color='#e67e22')
plt.title('Average Speed by Manufacturer (min 10 coasters)')
plt.xlabel('Average Speed')
plt.tight_layout()
plt.show()
plt.close()


In [ ]:
# C – Park specialties
park_stats = (roller_coasters
              .groupby('park')
              .agg(count=('name', 'count'),
                   avg_speed=('speed', 'mean'),
                   avg_height=('height', 'mean'),
                   avg_inv=('num_inversions', 'mean'))
              .query('count >= 5')
              .sort_values('count', ascending=False)
              .round(1))
print('=== Parks with most recorded coasters ===')
print(park_stats.head(10))

print('\n=== Parks with highest avg inversions (min 5 coasters) ===')
print(park_stats.sort_values('avg_inv', ascending=False).head(8))


In [ ]:
# D – Correlation heatmap
num_cols = ['speed', 'height', 'length', 'num_inversions']
corr = roller_coasters[num_cols].corr()
print(corr.round(3))

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap='RdYlBu_r', center=0, fmt='.2f',
            square=True, linewidths=0.5)
plt.title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.show()
plt.close()


## Alternate Code Approaches

In [ ]:
# Alternate 1 – seaborn line plot for ranking
def plot_coaster_ranking_sns(coaster_name, rankings_df, park_name=None):
    df = rankings_df[rankings_df['Name'] == coaster_name].copy()
    if park_name:
        df = df[df['Park'] == park_name]
    df = df.sort_values('Year of Rank')
    plt.figure(figsize=(8, 5))
    sns.lineplot(data=df, x='Year of Rank', y='Rank', marker='o', linewidth=2.5)
    plt.gca().invert_yaxis()
    plt.title(f'(seaborn) Ranking of {coaster_name}')
    plt.ylabel('Rank (1 = best)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close()

plot_coaster_ranking_sns('Fury 325', steel_rankings, park_name='Carowinds')


In [ ]:
# Alternate 2 – method chaining + query for manufacturer speed
top_speed_mfr = (
    roller_coasters
    .query('manufacturer != "na"')
    .groupby('manufacturer')
    .agg(n=('name', 'size'), avg_speed=('speed', 'mean'))
    .query('n >= 15')
    .sort_values('avg_speed', ascending=False)
    .head(6)
    .round(1)
)
print(top_speed_mfr)


In [ ]:
# Alternate 3 – pivot_table view of wood rankings (year × coaster for top names)
top_wood_names = (wood_rankings[wood_rankings['Rank'] <= 3]
                  ['Name'].unique())
pivot = (wood_rankings[wood_rankings['Name'].isin(top_wood_names)]
         .pivot_table(index='Year of Rank', columns='Name', values='Rank', aggfunc='min'))
print(pivot)
pivot.plot(marker='o', figsize=(10, 5))
plt.gca().invert_yaxis()
plt.title('Top Wood Coasters – Rank Pivot')
plt.ylabel('Rank')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()
plt.close()


## Simulation Section

Change parameters and observe the effect on conclusions.

In [ ]:
# Simulation 1 – different top-n
print('Top-3 wood coasters ever:')
print(sorted(wood_rankings[wood_rankings['Rank'] <= 3]['Name'].unique()))
print('\nTop-10 wood coasters ever:')
print(sorted(wood_rankings[wood_rankings['Rank'] <= 10]['Name'].unique())[:15], '...')

plot_top_n(steel_rankings, n=5, title_suffix=' (Steel, n=5)')


In [ ]:
# Simulation 2 – height filter threshold vs correlation
for thresh in [80, 120, 200]:
    sub = roller_coasters[(roller_coasters['height'] < thresh) &
                          roller_coasters['height'].notna() &
                          roller_coasters['speed'].notna()]
    c = sub[['height', 'speed']].corr().iloc[0, 1]
    print(f'Height < {thresh:3d} m → N={len(sub):4d}, corr(height,speed) = {c:.3f}')


In [ ]:
# Simulation 3 – synthetic popularity score with adjustable weights
def rank_by_synthetic_score(df, w_speed=0.4, w_height=0.3, w_inv=0.3, top_k=10):
    """Normalize speed, height, inversions and compute weighted score."""
    d = df[['name', 'park', 'speed', 'height', 'num_inversions']].dropna().copy()
    d = d[d['height'] < 150]
    for col in ['speed', 'height', 'num_inversions']:
        mn, mx = d[col].min(), d[col].max()
        d[col + '_n'] = (d[col] - mn) / (mx - mn + 1e-9)
    d['score'] = (w_speed * d['speed_n'] +
                  w_height * d['height_n'] +
                  w_inv * d['num_inversions_n'])
    return d.nlargest(top_k, 'score')[['name', 'park', 'speed', 'height', 'num_inversions', 'score']]

print('=== Weights (0.4 speed, 0.3 height, 0.3 inv) ===')
print(rank_by_synthetic_score(roller_coasters, 0.4, 0.3, 0.3).round(2).to_string(index=False))

print('\n=== Weights (0.2 speed, 0.5 height, 0.3 inv) – favor height ===')
print(rank_by_synthetic_score(roller_coasters, 0.2, 0.5, 0.3).round(2).to_string(index=False))

print('\n=== Weights (0.3 speed, 0.2 height, 0.5 inv) – favor inversions ===')
print(rank_by_synthetic_score(roller_coasters, 0.3, 0.2, 0.5).round(2).to_string(index=False))


## Key Insights from the Extended Analysis

1. **Rankings longevity** – *Millennium Force* (steel) and *Boulder Dash* / *Phoenix* (wood) dominate multi-year point totals; *Fury 325* rapidly rose to #1 after its 2015 debut.

2. **Height–Speed relationship** – Moderate positive correlation (~0.37 overall; higher when extreme outliers are removed). Hyper/giga coasters drive the upper-right of the scatter.

3. **Manufacturer focus** – RMC, B&M, S&S and Intamin produce the fastest average coasters; Vekoma and Zamperla dominate volume (family / mid-range).

4. **Seating & inversions** – Floorless and Wing seats are associated with higher average speeds; wooden coasters almost never invert (avg ~0.08 inversions vs ~0.8 for steel).

5. **Status** – ~77 % of catalogued coasters are still operating; the rest are closed, announced, under construction, etc.

6. **Simulation lesson** – Changing the height filter or the synthetic-score weights materially changes which coasters rank highest, illustrating sensitivity of “best coaster” claims to the chosen metric.

## Solution Complete

You have practiced ranking visualizations, statistical exploration, pattern mining, alternate coding styles, and parameter simulation.  
Compare your skeleton answers with this notebook and experiment further (different parks, different weight vectors, seaborn pairplots, etc.).